In [ ]:
#from ipaddress import ip_address
import pandas as pd
import numpy as np
import datetime as dt

from datetime import datetime, timedelta

from core.data import load_from_kaggle

/home/ulfgar/Lehrgänge/Projekte/portfolio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_link = "nokkyu/deutsche-bahn-db-delays" # replace with your dataset link from Kaggle 
destination = "../data/raw"
dataset_name = dataset_link.split("/")[-1]

files = load_from_kaggle(
    dataset_link=dataset_link, 
    destination=destination,
    )

Destination directory '../data/raw/deutsche-bahn-db-delays' already exists with files. Skipping download (replace=False).


In [3]:
df = pd.read_csv("/".join(["../data/raw/", dataset_name, files[0]]))
df.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,lat,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,2024-07-08 00:00:00,2024-07-08 00:01:00,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,NaN,2024-07-08 00:17:00,NaN,NaN,0,0,NaN,on_time,on_time
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,50.770202,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,NaN,NaN,0,0,NaN,on_time,on_time
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time


In [4]:
date_cols = ["arrival_plan", "departure_plan", "arrival_change", "departure_change"]
date_format = "%Y-%m-%d %H:%M:%S"

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format=date_format, errors='coerce')

print("\n--- Datentypen der Zeitspalten ---")
print(df[date_cols].dtypes)


--- Datentypen der Zeitspalten ---
arrival_plan        datetime64[ns]
departure_plan      datetime64[ns]
arrival_change      datetime64[ns]
departure_change    datetime64[ns]
dtype: object


In [5]:
# Zerlegung des Reisewegs
df['path_list'] = df['path'].str.split('|')

# Ableiten des Startbahnhofs
df['Start_Bahnhof'] = df['path_list'].str[0]

In [6]:
import re

# Bereinigung der Zielstation ('station' Spalte)
# Entfernt Klammern und Gleisangaben
df['station_clean'] = df['station'].str.replace(r'\s*\(.*?\)', '', regex=True)
df['station_clean'] = df['station_clean'].str.replace(r'\sGl\.\d+', '', regex=True)

# Bereinigung des Startbahnhofs
df['Start_Bahnhof_clean'] = df['Start_Bahnhof'].str.replace(r'\s*\(.*?\)', '', regex=True)
df['Start_Bahnhof_clean'] = df['Start_Bahnhof_clean'].str.replace(r'\sGl\.\d+', '', regex=True)

print("\n--- Bereinigte Bahnhofsbeispiele ---")
print(df[['station', 'station_clean', 'Start_Bahnhof_clean']].head(3))


--- Bereinigte Bahnhofsbeispiele ---
             station      station_clean Start_Bahnhof_clean
0         Aachen Hbf         Aachen Hbf         StolbergHbf
1         Aachen Hbf         Aachen Hbf                 NaN
2  Aachen-Rothe Erde  Aachen-Rothe Erde             HammHbf


In [7]:
import re

# Korrigierte Bereinigung (fügt das Leerzeichen zwischen Stadt und Hbf/West/etc. ein, falls es fehlt)

# Funktion, die zuerst Klammer und Gleis entfernt und dann fehlende Leerzeichen korrigiert
def clean_station_name(name):
    if pd.isna(name):
        return name
    
    # 1. Entferne Gleisangaben (z.B. " Gl.44")
    name = re.sub(r'\sGl\.\d+', '', name)
    
    # 2. Entferne Klammerzusätze (z.B. "(Rheinl)")
    name = re.sub(r'\s*\(.*?\)', '', name)
    
    # 3. Korrigiere fehlende Leerzeichen VOR Hbf/Süd/West/etc.
    # Füge ein Leerzeichen ein, wenn ein Großbuchstabe (außer A-Z oder Umlaute) direkt nach einem Kleinbuchstaben folgt.
    # Vereinfachte Korrektur: Füge ein Leerzeichen vor "Hbf", "Süd", "West" etc. ein, wenn kein Leerzeichen vorhanden ist
    name = re.sub(r'([a-z])(Hbf|Süd|West|Ost|Nord|ZOB|Flughafen)', r'\1 \2', name, flags=re.IGNORECASE)
    
    return name.strip()

# Anwendung der korrigierten Bereinigungsfunktion
df['station_clean'] = df['station'].apply(clean_station_name)
df['Start_Bahnhof_clean'] = df['Start_Bahnhof'].apply(clean_station_name)

print("\n--- Korrigierte Bahnhofsbeispiele ---")
print(df[['station', 'station_clean', 'Start_Bahnhof_clean']].head(3))


--- Korrigierte Bahnhofsbeispiele ---
             station      station_clean Start_Bahnhof_clean
0         Aachen Hbf         Aachen Hbf        Stolberg Hbf
1         Aachen Hbf         Aachen Hbf                 NaN
2  Aachen-Rothe Erde  Aachen-Rothe Erde            Hamm Hbf


In [8]:
# Verspätung bei Ankunft
df["arrival_delay_calc"] = (
    df["arrival_change"] - df["arrival_plan"]
).dt.total_seconds() / 60

# Verspätung bei Abfahrt
df["departure_delay_calc"] = (
    df["departure_change"] - df["departure_plan"]
).dt.total_seconds() / 60

In [9]:
df['synthetic'] = False

print(f"\nVerspätungen berechnet und 'synthetic' Spalte erstellt.")


Verspätungen berechnet und 'synthetic' Spalte erstellt.


In [10]:
# 4.1 Basis für die Erzeugung: Züge mit Leipzig-Bezug (oder einfach die ersten 500, falls keine vorhanden)
leipzig_base = df[
    (df['station'].str.contains('Leipzig', na=False)) |
    (df['Start_Bahnhof_clean'].str.contains('Leipzig', na=False))
].copy()

# Stichprobe ziehen (mit Wiederholung, falls Basis < 500)
if leipzig_base.shape[0] < 500:
    df_leipzig_verspaetung = leipzig_base.sample(n=500, replace=True, random_state=42).copy()
else:
    df_leipzig_verspaetung = leipzig_base.sample(n=500, replace=False, random_state=42).copy()

# 4.2 Daten manipulieren
sim_date = dt.datetime(2024, 7, 8)
df_leipzig_verspaetung['station_clean'] = 'Leipzig Hbf'
df_leipzig_verspaetung['synthetic'] = True # Markierung

for index in df_leipzig_verspaetung.index:
    delay_min = np.random.randint(1, 91)
    
    for t_plan, t_change, t_delay_calc in [
        ('arrival_plan', 'arrival_change', 'arrival_delay_calc'),
        ('departure_plan', 'departure_change', 'departure_delay_calc')
    ]:
        if pd.notna(df_leipzig_verspaetung.loc[index, t_plan]):
            time_part = df_leipzig_verspaetung.loc[index, t_plan].time()
            # Setze das Datum auf 2024-07-08
            df_leipzig_verspaetung.loc[index, t_plan] = dt.datetime.combine(sim_date.date(), time_part)
            # Berechne die geänderte Zeit
            df_leipzig_verspaetung.loc[index, t_change] = df_leipzig_verspaetung.loc[index, t_plan] + dt.timedelta(minutes=delay_min)
            df_leipzig_verspaetung.loc[index, t_delay_calc] = delay_min

In [11]:
# Sicherstellen, dass keine alten synthetischen Daten vorhanden sind, falls der Block wiederholt wird
df = df[df['synthetic'] == False].copy()

# Zusammenführen der DataFrames
df = pd.concat([df, df_leipzig_verspaetung], ignore_index=True)

print(f"\nDatenzusammenführung abgeschlossen. Gesamtzeilen: {len(df)}. Synthetische Zeilen: {len(df[df['synthetic'] == True])}.")


Datenzusammenführung abgeschlossen. Gesamtzeilen: 2061857. Synthetische Zeilen: 500.


In [12]:
def analyze_missed_and_next_connections(delayed_trains_df, departure_trains_df, min_transfer_minutes=5):
    """
    Findet für jeden verspäteten Zug den verpassten Anschluss und den nächsten Anschluss.
    """
    results = []
    min_transfer_time = pd.Timedelta(minutes=min_transfer_minutes)
    departure_trains_df = departure_trains_df.sort_values(by='departure_plan').reset_index(drop=True)

    for index, arr_row in delayed_trains_df.iterrows():
        arr_plan = arr_row['arrival_plan']
        arr_changed = arr_row['arrival_change']
        
        # 1. Verpasste Züge: Planmäßig erreichbar (Ankunft_Plan + Puffer) ABER Abfahrt vor tatsächlicher Ankunft + Puffer
        potentially_reachable = departure_trains_df[
            departure_trains_df['departure_plan'] >= arr_plan + min_transfer_time
        ]
        missed_candidates = potentially_reachable[
            potentially_reachable['departure_plan'] < arr_changed + min_transfer_time
        ]
        
        # 2. Nächstmöglicher Zug
        earliest_departure_plan = arr_changed + min_transfer_time
        next_candidates = departure_trains_df[
            departure_trains_df['departure_plan'] >= earliest_departure_plan
        ]

        missed_conn = missed_candidates.iloc[-1] if not missed_candidates.empty else None
        next_conn = next_candidates.iloc[0] if not next_candidates.empty else None
        
        # Datenextraktion
        missed_data = {
            'Verpasst_Kat': missed_conn['category'] if missed_conn is not None else 'N/A',
            'Verpasst_Ziel': missed_conn['path_list'][-1] if missed_conn is not None and isinstance(missed_conn['path_list'], list) else (missed_conn['station'] if missed_conn is not None else 'N/A'),
            'Verpasst_Abfahrt_Plan': missed_conn['departure_plan'] if missed_conn is not None else pd.NaT,
        }
        next_data = {
            'Naechste_Abfahrt': next_conn['departure_plan'] if next_conn is not None else pd.NaT,
            'Naechste_Wartezeit_Min': ((next_conn['departure_plan'] - arr_changed).total_seconds() / 60) if next_conn is not None else np.nan,
        }

        results.append({
            'Verspätung_Min': arr_row['arrival_delay_calc'],
            'Verspätung_Kat': arr_row['category'],
            **missed_data,
            **next_data
        })

    return pd.DataFrame(results)

In [13]:
# Filterung der Daten für die Funktionsausführung
df_leipzig_synthetic = df[
    (df['station_clean'] == 'Leipzig Hbf') & 
    (df['synthetic'] == True) & 
    (df['arrival_delay_calc'].notna())
].copy()

df_leipzig_departures = df[
    (df['Start_Bahnhof_clean'] == 'Leipzig Hbf') &
    (df['departure_plan'].dt.date == pd.to_datetime('2024-07-08').date()) &
    (df['synthetic'] == False) 
].copy()

# Ausführung der Analyse
missed_connections_df = analyze_missed_and_next_connections(df_leipzig_synthetic, df_leipzig_departures)

# Sortierung und Darstellung der Top 10
missed_connections_df_sorted = missed_connections_df.sort_values(by='Naechste_Wartezeit_Min', ascending=False)

final_missed_output = missed_connections_df_sorted[[
    'Verspätung_Min', 
    'Verpasst_Kat', 
    'Verpasst_Ziel', 
    'Naechste_Wartezeit_Min'
]].head(10)

final_missed_output.columns = [
    'Verspätung (Min)', 
    'Verpasst Kat.', 
    'Verpasst Ziel', 
    'Wartezeit (Min)'
]

print("\n### Top 10 Verpasste Anschlüsse und Wartezeiten in Leipzig Hbf (Ergebnis) 📊")
print(final_missed_output)


### Top 10 Verpasste Anschlüsse und Wartezeiten in Leipzig Hbf (Ergebnis) 📊
     Verspätung (Min) Verpasst Kat. Verpasst Ziel  Wartezeit (Min)
289              61.0             4     Mockrehna            156.0
37                9.0           N/A           N/A            150.0
274              71.0             4     Mockrehna            146.0
32               83.0             4     Mockrehna            138.0
170              82.0             4     Mockrehna            122.0
17               90.0             4     Mockrehna             99.0
34               64.0           N/A           N/A             70.0
401              85.0           N/A           N/A             53.0
72               79.0           N/A           N/A             46.0
179              10.0           N/A           N/A             40.0


Versuch die txt Dateien von mdv.de zu integrieren

In [14]:

# Erstellen Sie den Pfad zum Datenordner
mdv_data_dir = "../data/raw/" # Passen Sie dies bei Bedarf an

# --- A) Einlesen der Stoppzeiten ---
# Wir müssen die Zeitspalten separat behandeln, da sie kein Datum enthalten.
try:
    df_stop_times = pd.read_csv(
        f"{mdv_data_dir}stop_times.txt",  # Beispiel-Dateiname
        sep=',', 
        dtype={'stop_id': str, 'trip_id': str}
    )

    # Funktion zur Konvertierung von HH:MM:SS in Zeitstempel (mit fiktivem Datum 2024-07-08)
    def parse_time_to_datetime(time_str):
        if pd.isna(time_str):
            return pd.NaT
        # Das MDV-Format kann Stunden > 24 haben (für Fahrten über Mitternacht).
        # Wir müssen hier eine komplexe Logik implementieren, um das Datum korrekt zu setzen,
        # was wir hier vereinfachen, indem wir nur die Uhrzeit parsen.
        try:
            return pd.to_datetime(time_str, format='%H:%M:%S').time()
        except ValueError:
            # Behandlung von Zeiten > 23:59:59 (z.B. 25:00:00)
            return pd.NaT 

    # Beispiel-Konvertierung für die kritischen Zeitspalten
    # ANNAHME: Spalten heißen 'arrival_time' und 'departure_time'
    df_stop_times['arrival_time'] = df_stop_times['arrival_time'].apply(parse_time_to_datetime)
    df_stop_times['departure_time'] = df_stop_times['departure_time'].apply(parse_time_to_datetime)

    print(f"\nstop_times.txt erfolgreich geladen: {len(df_stop_times)} Zeilen.")

except FileNotFoundError:
    print("\nFEHLER: 'stop_times.txt' nicht gefunden. Bitte Dateinamen prüfen.")


stop_times.txt erfolgreich geladen: 1583706 Zeilen.


In [15]:
# ANNAHME: Die Datei liegt im gleichen Pfad und verwendet Semikolons
mdv_data_dir = "../data/raw/" # Passen Sie dies bei Bedarf an

# --- A) Einlesen der Haltestellen ---
try:
    df_stops = pd.read_csv(
        f"{mdv_data_dir}stops.txt",
        sep=',', 
        dtype={'stop_id': str} # Wichtig: IDs als Strings behandeln
    )
    print(f"stops.txt erfolgreich geladen: {len(df_stops)} Zeilen.")

    # ANNAHME: Spaltennamen sind 'stop_id' und 'stop_name'
    print(f"\nKopfzeile von stops.txt:\n{df_stops.head()}")

except FileNotFoundError:
    print("\nFEHLER: 'stops.txt' nicht gefunden. Bitte Dateinamen prüfen.")

stops.txt erfolgreich geladen: 5299 Zeilen.

Kopfzeile von stops.txt:
     stop_id                stop_name   stop_lat   stop_lon
0  000000145      Leipzig, Arcus Park  51.361549  12.446576
1  000000152        Leipzig, Bergstr.  51.340535  12.402015
2  000000154  Leipzig, Breitscheidhof  51.351846  12.286913
3  000000163     Leipzig, Bremer Str.  51.376178  12.365034
4  000000164    Leipzig, Dankwartstr.  51.299188  12.392814


In [16]:
# 2.1 Bereinigung der MDV-Namen (Funktion wie zuvor definiert)
# WICHTIG: Die Funktion clean_station_name muss im Speicher verfügbar sein
def clean_mdv_stop_name(name):
    if pd.isna(name):
        return name
    name = str(name)
    # Entferne Zusätze in Klammern (oft Ortszusätze)
    name = re.sub(r'\s*\(.*?\)', '', name)
    # Entferne typische Haltestellenzusätze (z.B. Hst, Bf, ZOB, Bhf)
    name = re.sub(r'\s(Hst|Bf|ZOB|Bhf)\b', '', name)
    return name.strip()

# Bereinigung der MDV-Stoppnamen
df_stops['stop_name_clean'] = df_stops['stop_name'].apply(clean_mdv_stop_name)

# 2.2 Identifizierung der ID für Leipzig Hbf
# Wir suchen nach allen IDs, die Hbf im Namen haben
leipzig_hbf_mdv = df_stops[
    df_stops['stop_name_clean'].str.contains('Hbf', na=False, case=False) &
    df_stops['stop_name_clean'].str.contains('Leipzig', na=False, case=False)
].copy()

LEIPZIG_HBF_STOP_ID = None

if not leipzig_hbf_mdv.empty:
    # Wir nehmen die ID mit dem kürzesten Namen, da diese oft der Hauptknoten ist
    # Beispiel: "Leipzig Hbf" vs. "Leipzig Hbf (Bahnsteig 1)"
    leipzig_hbf_mdv['name_length'] = leipzig_hbf_mdv['stop_name_clean'].str.len()
    LEIPZIG_HBF_STOP_ID = leipzig_hbf_mdv.sort_values(by='name_length')['stop_id'].iloc[0]
    
    print(f"\nIdentifizierte MDV Stop ID für Leipzig Hbf: **{LEIPZIG_HBF_STOP_ID}**")
    print("Zugehöriger Stoppname:", leipzig_hbf_mdv[leipzig_hbf_mdv['stop_id'] == LEIPZIG_HBF_STOP_ID]['stop_name'].iloc[0])
else:
    print("\nFEHLER: 'Leipzig Hbf' konnte in stops.txt nicht eindeutig identifiziert werden. Bitte manuelle Prüfung.")


Identifizierte MDV Stop ID für Leipzig Hbf: **008010205**
Zugehöriger Stoppname: Leipzig Hbf


In [17]:
# ANNAHME: Die Datei liegt im gleichen Pfad und verwendet Semikolons
mdv_data_dir = "../data/raw/" 

# --- A) Einlesen der Fahrten (Trips) ---
try:
    df_trips = pd.read_csv(
        f"{mdv_data_dir}trips.txt",
        sep=',', 
        dtype={'route_id': str, 'trip_id': str}
    )
    print(f"\ntrips.txt erfolgreich geladen: {len(df_trips)} Zeilen.")

    # ANNAHME: Spaltennamen sind 'trip_id', 'route_id', 'direction_id' und 'trip_headsign'
    print(f"\nKopfzeile von trips.txt:\n{df_trips.head()}")

except FileNotFoundError:
    print("\nFEHLER: 'trips.txt' nicht gefunden. Bitte Dateinamen prüfen.")


trips.txt erfolgreich geladen: 79863 Zeilen.

Kopfzeile von trips.txt:
      route_id  service_id trip_id trip_headsign  trip_short_name  \
0  800413RB113         460       1   Belgershain              NaN   
1  800413RB113        1659       2      Geithain              NaN   
2  800413RB113         460       3   Belgershain              NaN   
3  800413RB113        1238       4      Geithain              NaN   
4  800413RB113         270       5   Belgershain              NaN   

   direction_id  
0             1  
1             1  
2             1  
3             1  
4             1  


In [18]:
# Die MDV-Stop-ID für Leipzig Hbf
LEIPZIG_HBF_STOP_ID = "008010205"

def analyze_mdv_connections(delayed_trains_df, df_stop_times, df_trips, stop_id, min_transfer_minutes=5):
    """
    Findet für jede DB-Verspätung den nächstmöglichen Anschluss im MDV-Fahrplan (stop_times).
    """
    results = []
    min_transfer_time = pd.Timedelta(minutes=min_transfer_minutes)
    
    # 1. Vorfilterung der MDV-Abfahrten von Leipzig Hbf
    mdv_departures = df_stop_times[df_stop_times['stop_id'] == stop_id].copy()
    
    # Sicherstellen, dass die Uhrzeit-Spalte existiert (Annahme: 'departure_time')
    if 'departure_time' not in mdv_departures.columns:
        print("FEHLER: 'departure_time' fehlt in df_stop_times.")
        return pd.DataFrame(results)

    # 2. Schleife durch die 50 simulierten DB-Ankünfte
    for index, arr_row in delayed_trains_df.iterrows():
        arr_changed = arr_row['arrival_change'] # Tatsächliche Ankunft (Datum + Uhrzeit)
        earliest_departure_dt = arr_changed + min_transfer_time # Früheste mögliche Abfahrt
        
        # Extrahieren der Uhrzeit und des Datums
        target_date = earliest_departure_dt.date()
        target_time = earliest_departure_dt.time()

        # 3. MDV-Verbindungen für den aktuellen Tag (oder den nächsten Tag bei Mitternacht) finden
        
        # Konvertiere die MDV-Uhrzeiten in volle Datums-Zeitstempel (für den Vergleich)
        # Wir müssen jeden MDV-Abfahrtszeitpunkt mit dem Zieldatum kombinieren
        
        # Konvertierung der MDV-Abfahrtszeiten in datetime-Objekte am Zieldatum
        mdv_departures['combined_departure_dt'] = mdv_departures['departure_time'].apply(
            lambda t: datetime.combine(target_date, t) if pd.notna(t) else pd.NaT
        )
        
        # Berücksichtigung von Fahrten, die erst am nächsten Tag starten (z.B. 00:05 Uhr, aber Verspätung endet um 23:55 Uhr)
        # Wenn die kombinierte Zeit vor der Zielzeit liegt (z.B. 01:00 Uhr am 2024-07-08 liegt vor 23:55 Uhr am 2024-07-08),
        # fügen wir einen Tag hinzu. DIES IST VEREINFACHT, da GTFS-Daten oft >24h Zeiten enthalten,
        # was hier nicht modelliert werden kann. Wir verlassen uns auf die `HH:MM:SS` Zeiten und sortieren.
        
        # WICHTIG: Filtern auf Abfahrten, die NACH der frühestmöglichen Abfahrt liegen
        next_mdv_candidates = mdv_departures[
            mdv_departures['combined_departure_dt'] >= earliest_departure_dt
        ].sort_values(by='combined_departure_dt')
        
        # Suche nach dem nächstmöglichen Anschluss
        next_mdv_conn = next_mdv_candidates.iloc[0] if not next_mdv_candidates.empty else None
        
        if next_mdv_conn is not None:
            # 4. Wartezeit und Ziel ermitteln
            wait_time = (next_mdv_conn['combined_departure_dt'] - arr_changed).total_seconds() / 60
            
            # Verknüpfung mit trips.txt für das Ziel (trip_headsign)
            trip_info = df_trips[df_trips['trip_id'] == next_mdv_conn['trip_id']]
            mdv_ziel = trip_info['trip_headsign'].iloc[0] if not trip_info.empty else 'Unbekanntes MDV-Ziel'
            mdv_route = trip_info['route_id'].iloc[0] if not trip_info.empty else 'Unbekannte Route'

            results.append({
                'Verspätung_Min_DB': arr_row['arrival_delay_calc'],
                'Ankunft_Geändert_DB': arr_changed,
                'MDV_Abfahrt_Plan': next_mdv_conn['combined_departure_dt'],
                'MDV_Zielbahnhof': mdv_ziel,
                'MDV_Route': mdv_route,
                'Wartezeit_MDV_Min': wait_time
            })
        else:
            results.append({
                'Verspätung_Min_DB': arr_row['arrival_delay_calc'],
                'Ankunft_Geändert_DB': arr_changed,
                'MDV_Abfahrt_Plan': pd.NaT,
                'MDV_Zielbahnhof': 'Kein Anschluss gefunden',
                'MDV_Route': 'N/A',
                'Wartezeit_MDV_Min': np.nan
            })

    return pd.DataFrame(results)

# --- Ausführung der Analyse ---
# Sicherstellen, dass df_stop_times die 'departure_time' als dt.time() Objekt enthält
# (Dies wurde im vorherigen Schritt implizit gemacht. Falls nicht, muss es hier erfolgen.)
mdv_connection_analysis = analyze_mdv_connections(
    df_leipzig_synthetic, 
    df_stop_times, 
    df_trips, 
    LEIPZIG_HBF_STOP_ID
)

In [19]:
# Die MDV-Stop-ID für Leipzig Hbf
LEIPZIG_HBF_STOP_ID = "008010205"

def analyze_mdv_connections(delayed_trains_df, df_stop_times, df_trips, stop_id, min_transfer_minutes=5):
    """
    Findet für jede DB-Verspätung den nächstmöglichen Anschluss im MDV-Fahrplan (stop_times).
    """
    results = []
    min_transfer_time = pd.Timedelta(minutes=min_transfer_minutes)
    
    mdv_departures = df_stop_times[df_stop_times['stop_id'] == stop_id].copy()
    
    # 2. Schleife durch die 50 simulierten DB-Ankünfte
    for index, arr_row in delayed_trains_df.iterrows():
        arr_changed = arr_row['arrival_change']
        earliest_departure_dt = arr_changed + min_transfer_time
        target_date = earliest_departure_dt.date()

        # Konvertierung der MDV-Uhrzeiten in volle Datums-Zeitstempel am Zieldatum
        mdv_departures['combined_departure_dt'] = mdv_departures['departure_time'].apply(
            lambda t: dt.datetime.combine(target_date, t) if pd.notna(t) else pd.NaT
        )
        
        # WICHTIG: Filtern auf Abfahrten, die NACH der frühestmöglichen Abfahrt liegen
        next_mdv_candidates = mdv_departures[
            mdv_departures['combined_departure_dt'] >= earliest_departure_dt
        ].sort_values(by='combined_departure_dt')
        
        next_mdv_conn = next_mdv_candidates.iloc[0] if not next_mdv_candidates.empty else None
        
        if next_mdv_conn is not None:
            # 4. Wartezeit und Ziel ermitteln
            wait_time = (next_mdv_conn['combined_departure_dt'] - arr_changed).total_seconds() / 60
            
            # Verknüpfung mit trips.txt für das Ziel (trip_headsign)
            trip_info = df_trips[df_trips['trip_id'] == next_mdv_conn['trip_id']]
            mdv_ziel = trip_info['trip_headsign'].iloc[0] if not trip_info.empty else 'Unbekanntes MDV-Ziel'
            mdv_route = trip_info['route_id'].iloc[0] if not trip_info.empty else 'Unbekannte Route'

            results.append({
                'DB_Verspätung_Min': arr_row['arrival_delay_calc'],
                'DB_Ankunft_Geändert': arr_changed,
                'MDV_Zielbahnhof': mdv_ziel,
                'Wartezeit_MDV_Min': wait_time
            })
        else:
            results.append({
                'DB_Verspätung_Min': arr_row['arrival_delay_calc'],
                'DB_Ankunft_Geändert': arr_changed,
                'MDV_Zielbahnhof': 'Kein Anschluss gefunden',
                'Wartezeit_MDV_Min': np.nan
            })

    return pd.DataFrame(results)

# --- Ausführung der Analyse ---
mdv_connection_analysis = analyze_mdv_connections(
    df_leipzig_synthetic, 
    df_stop_times, 
    df_trips, 
    LEIPZIG_HBF_STOP_ID
)

print("MDV-Anschlussanalyse abgeschlossen. Wartezeiten berechnet.")

MDV-Anschlussanalyse abgeschlossen. Wartezeiten berechnet.


In [20]:
# Füge die MDV-Spalten zum DB-Analyse-DF hinzu
comparison_df = pd.concat([
    missed_connections_df.reset_index(drop=True), # DB-Analyse (verpasster Zug + nächster DB-Zug)
    mdv_connection_analysis[['MDV_Zielbahnhof', 'Wartezeit_MDV_Min']] # MDV-Analyse
], axis=1)

# Berechne die Differenz in der Wartezeit (Positiv = MDV ist besser)
comparison_df['DB_Wartezeit_Min'] = comparison_df['Naechste_Wartezeit_Min']
comparison_df['Vorteil_MDV_Min'] = comparison_df['DB_Wartezeit_Min'] - comparison_df['Wartezeit_MDV_Min']

In [21]:
# Sortiere nach dem größten Wartezeit-Vorteil des MDV (DB Wartezeit - MDV Wartezeit)
comparison_df_sorted = comparison_df.sort_values(
    by='Vorteil_MDV_Min', 
    ascending=False
)

final_comparison_output = comparison_df_sorted[[
    'Verspätung_Min', 
    'DB_Wartezeit_Min', 
    'Wartezeit_MDV_Min', 
    'Vorteil_MDV_Min',
    'Verpasst_Ziel', 
    'MDV_Zielbahnhof'
]].head(10)

final_comparison_output.columns = [
    'Verspätung (Min)', 
    'Wartezeit DB (Min)', 
    'Wartezeit MDV (Min)', 
    'MDV Vorteil (Min)', 
    'Verpasst Ziel', 
    'MDV Ziel (Nächst)'
]

print("\n### Top 10: Wartezeit-Vergleich DB vs. MDV (Alternative Routen) 🚂🚌")
print(final_comparison_output)


### Top 10: Wartezeit-Vergleich DB vs. MDV (Alternative Routen) 🚂🚌
     Verspätung (Min)  Wartezeit DB (Min)  Wartezeit MDV (Min)  \
289              61.0               156.0                  6.0   
17               90.0                99.0                 51.0   
32               83.0               138.0                 90.0   
274              71.0               146.0                 98.0   
34               64.0                70.0                 22.0   
170              82.0               122.0                 74.0   
401              85.0                53.0                  5.0   
37                9.0               150.0                102.0   
72               79.0                46.0                  9.0   
179              10.0                40.0                  5.0   

     MDV Vorteil (Min) Verpasst Ziel MDV Ziel (Nächst)  
289              150.0     Mockrehna      Grimma ob Bf  
17                48.0     Mockrehna       Leipzig Hbf  
32                48.0     Mockreh

In [22]:
mean_db = comparison_df['DB_Wartezeit_Min'].mean()
mean_mdv = comparison_df['Wartezeit_MDV_Min'].mean()

print("\n--- Statistische Zusammenfassung der Wartezeiten ---")
print(f"Durchschnittliche Wartezeit auf den nächsten DB-Anschluss: **{mean_db:.2f} Min**")
print(f"Durchschnittliche Wartezeit auf den nächsten MDV-Anschluss: **{mean_mdv:.2f} Min**")

if mean_mdv < mean_db:
    print(f"Fazit: Im Durchschnitt bietet der MDV-Fahrplan eine kürzere Wartezeit von {(mean_db - mean_mdv):.2f} Minuten.")
else:
    print(f"Fazit: Im Durchschnitt bietet der DB-Anschluss die kürzere Wartezeit.")


--- Statistische Zusammenfassung der Wartezeiten ---
Durchschnittliche Wartezeit auf den nächsten DB-Anschluss: **8.08 Min**
Durchschnittliche Wartezeit auf den nächsten MDV-Anschluss: **8.66 Min**
Fazit: Im Durchschnitt bietet der DB-Anschluss die kürzere Wartezeit.


detaillierte Wartezeitverteilung nach Zugkategorie

In [24]:
# --- VORAUSSETZUNG: Erneute Ausführung der 500er Simulation ist erfolgt ---

# 1. Sicherstellen, dass die DB- und MDV-Ergebnisse die gleiche Reihenfolge/Anzahl haben (via Index)
# Wir setzen die Indizes zurück, um die Zeilen 1:1 zu verknüpfen
df_db_temp = missed_connections_df.reset_index(drop=True)
df_mdv_temp = mdv_connection_analysis.reset_index(drop=True)

# 2. Zusammenführung der Analysen in einem DataFrame
comparison_df = pd.concat([
    df_db_temp[[
        'Verspätung_Min', 'Naechste_Wartezeit_Min', 'Verpasst_Ziel', 'Verpasst_Kat'
    ]].rename(columns={'Naechste_Wartezeit_Min': 'DB_Wartezeit_Min', 'Verspätung_Min': 'DB_Verspätung_Min'}),
    
    df_mdv_temp[[
        'DB_Ankunft_Geändert', 'Wartezeit_MDV_Min', 'MDV_Zielbahnhof'
    ]]
], axis=1)

# 3. Berechnung der Differenz und des Vorteils
comparison_df['Vorteil_MDV_Min'] = comparison_df['DB_Wartezeit_Min'] - comparison_df['Wartezeit_MDV_Min']

# --- A) Wartezeit nach Tageszeit (jetzt ohne KeyError) ---

def categorize_time_of_day(dt_object):
    """Kategorisiert die Ankunftszeit in Tagesphasen."""
    hour = dt_object.hour
    if 5 <= hour < 9:
        return '1. Früh/Pendler (05-09h)'
    elif 9 <= hour < 16:
        return '2. Tag/Mittag (09-16h)'
    elif 16 <= hour < 20:
        return '3. Abend/Pendler (16-20h)'
    else:
        return '4. Nacht (20-05h)'

# Anwendung der korrigierten Spalte
comparison_df['Tageszeit'] = comparison_df['DB_Ankunft_Geändert'].apply(categorize_time_of_day)

# Gruppierung nach Tageszeit
time_of_day_analysis = comparison_df.groupby('Tageszeit').agg(
    Wartezeit_DB_Mean=('DB_Wartezeit_Min', 'mean'),
    Wartezeit_MDV_Mean=('Wartezeit_MDV_Min', 'mean'),
    MDV_Vorteil_Mean=('Vorteil_MDV_Min', 'mean'),
    Anzahl=('DB_Verspätung_Min', 'count')
).reset_index().sort_values(by='Tageszeit')

print("\n### Wartezeit (Minuten) im Durchschnitt nach Tageszeit 🌞🌙")
time_of_day_analysis.columns = [
    'Tageszeit', 'DB (Ø Min)', 'MDV (Ø Min)', 'MDV Vorteil (Ø Min)', 'Anzahl'
]
print(time_of_day_analysis)

# --- B) Wartezeit nach Zugkategorie (wie zuvor geplant) ---
category_analysis = comparison_df.groupby('Verpasst_Kat').agg(
    Wartezeit_DB_Median=('DB_Wartezeit_Min', 'median'),
    Wartezeit_MDV_Median=('Wartezeit_MDV_Min', 'median'),
    MDV_Vorteil_Mean=('Vorteil_MDV_Min', 'mean'),
    Anzahl=('DB_Verspätung_Min', 'count')
).reset_index().sort_values(by='Anzahl', ascending=False)

print("\n### Wartezeit (Median) nach verpasster Zugkategorie 🚄")
category_analysis.columns = [
    'Verpasst Zug Kat.', 'DB (Median Min)', 'MDV (Median Min)', 'MDV Vorteil (Ø Min)', 'Anzahl'
]
print(category_analysis)


### Wartezeit (Minuten) im Durchschnitt nach Tageszeit 🌞🌙
                   Tageszeit  DB (Ø Min)  MDV (Ø Min)  MDV Vorteil (Ø Min)  \
0   1. Früh/Pendler (05-09h)    5.680000     6.960000            -1.280000   
1     2. Tag/Mittag (09-16h)    5.728916     7.138554            -1.409639   
2  3. Abend/Pendler (16-20h)    5.427083     7.916667            -2.489583   
3          4. Nacht (20-05h)   17.054348    12.915094             4.315217   

   Anzahl  
0      75  
1     166  
2      96  
3     106  

### Wartezeit (Median) nach verpasster Zugkategorie 🚄
  Verpasst Zug Kat.  DB (Median Min)  MDV (Median Min)  MDV Vorteil (Ø Min)  \
3                 5              5.0               7.0            -2.445860   
2                 4              6.0               6.0             1.104938   
1                 3              6.0               6.0            -1.168831   
0                 2              5.0               7.0            -1.695652   
4               N/A             28.5    

Maßnahmenkatalog für Leipzig Hbf

AnkunftWartezeit auf nächsten DB-ZugStrategie:

05:00 – 20:00 Uhr< 10 MinutenWartezeit auf DB-Anschluss
05:00 – 20:00 Uhr≥ 10 MinutenUmstieg auf MDV-Nahverkehr
20:00 – 05:00 UhrImmerUmstieg auf MDV-Nahverkehr

